# Comparative Legal Clause Classification

This notebook builds a reproducible NLP coursework pipeline for legal clause classification using LEDGAR as the main dataset. The input is a legal clause or provision text, and the output is a predicted clause category.

The notebook compares four families of approaches:

- dummy baselines for lower-bound context
- classical sparse-text models using TF-IDF features
- an optional fine-tuned transformer classifier
- an optional Qwen2.5-Instruct prompting baseline

A small human-in-the-loop review prototype is included only as an illustrative extension. It is not a legal advice system and does not claim to assess legal risk.

The implementation lives in `modules/`, while this notebook acts as the orchestration, reporting, and display layer. Results are generated only by running the cells; no metrics are inserted manually.


## 1. Colab Setup, Imports, and Configuration

This stage prepares the runtime so the same notebook can run locally or in Google Colab. In Colab, upload, unzip, sync, or clone the whole project folder, not just this notebook.



Importing Libraries and Modules

In [ ]:
from pathlib import Path

import importlib.util
import os
import subprocess
import sys
import numpy as np
import pandas as pd

import json
import math
import re

import pandas as pd
from datetime import datetime, timezone
from IPython.display import display

File Setup

In [ ]:
PROJECT_ROOT_OVERRIDE = os.environ.get("LEDGAR_PROJECT_ROOT", "").strip()
AUTO_MOUNT_GOOGLE_DRIVE = True
INSTALL_REQUIREMENTS_IN_COLAB = True


def running_in_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or importlib.util.find_spec("google.colab") is not None


IN_COLAB = running_in_colab()

In [ ]:

if IN_COLAB:
    print("Google Colab runtime detected.")

if IN_COLAB and AUTO_MOUNT_GOOGLE_DRIVE:
    try:
        if Path("/content/drive/MyDrive").exists():
            print("Google Drive is already available.")
        else:
            from google.colab import drive

            drive.mount("/content/drive")
    except Exception as exc:
        print(f"Google Drive mount skipped/failed: {type(exc).__name__}: {exc}")


def looks_like_project_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "modules").is_dir()


def project_root_candidates_near(path: Path) -> list[Path]:
    path = path.expanduser()
    candidates = [path, *path.parents]
    if path.exists() and path.is_dir():
        for pattern in (
            "pyproject.toml",
            "*/pyproject.toml",
            "*/*/pyproject.toml",
            "*/*/*/pyproject.toml",
        ):
            candidates.extend(pyproject.parent for pyproject in path.glob(pattern))
    deduped = []
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            resolved = candidate
        if resolved not in seen:
            deduped.append(resolved)
            seen.add(resolved)
    return deduped


def parent_search(start: Path) -> Path | None:
    for candidate in project_root_candidates_near(start):
        if looks_like_project_root(candidate):
            return candidate
    return None


def common_colab_candidates() -> list[Path]:
    candidates = [
        Path("/content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing"),
    ]
    for base in (Path("/content"), Path("/content/drive/MyDrive")):
        if base.exists():
            for pattern in (
                "Natural-Language-Processing",
                "*/Natural-Language-Processing",
                "*/*/Natural-Language-Processing",
                "*/*/*/Natural-Language-Processing",
            ):
                candidates.extend(base.glob(pattern))
    return candidates


def find_notebook_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE:
        override = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        for candidate in project_root_candidates_near(override):
            if looks_like_project_root(candidate):
                if candidate != override:
                    print(f"PROJECT_ROOT_OVERRIDE pointed to a parent folder; using nested project root: {candidate}")
                return candidate
        raise FileNotFoundError(
            f"PROJECT_ROOT_OVERRIDE does not contain pyproject.toml and modules/, and no nested project root was found under it: {override}\n"
            "Check the Drive folder path, or run this diagnostic: list(Path('/content/drive/MyDrive').glob('**/pyproject.toml'))"
        )

    root = parent_search(Path.cwd())
    if root is not None:
        return root

    if IN_COLAB:
        for candidate in common_colab_candidates():
            if candidate.exists() and looks_like_project_root(candidate):
                return candidate.resolve()

    raise FileNotFoundError(
        "Could not find the project root containing pyproject.toml and modules/.\n"
        "In Colab, upload or clone the whole repository, then set PROJECT_ROOT_OVERRIDE "
        "near the top of this cell to that folder. Current working directory: "
        f"{Path.cwd()}"
    )

Google Colab runtime detected.
Mounted at /content/drive
Project root: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing
Colab runtime: True
Raw LEDGAR directory: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/data/raw/lexglue_ledgar
Results directory: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/results
Device: cuda
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
# Find the project root and add it to sys.path so that imports work, even if the notebook is opened in a subfolder or outside the project.
PROJECT_ROOT = find_notebook_project_root()
os.environ["LEDGAR_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# List of (import_name, pip_name) for packages commonly used in notebooks. pip_name can be None if it's the same as import_name.
REQUIRED_NOTEBOOK_PACKAGES = [
    ("pandas", "pandas"),
    ("datasets", "datasets"),
    ("huggingface_hub", "huggingface_hub"),
    ("sklearn", "scikit-learn"),
    ("joblib", "joblib"),
    ("matplotlib", "matplotlib"),
]

# In Colab, install all requirements from requirements-colab.txt if any are missing, to avoid multiple pip installs.
def ensure_notebook_package(import_name: str, pip_name: str | None = None) -> None:
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

# Check for missing imports before installing requirements in Colab, to avoid unnecessary pip installs and speed up notebook startup.
missing_imports = [name for name, _ in REQUIRED_NOTEBOOK_PACKAGES if importlib.util.find_spec(name) is None]
requirements_path = PROJECT_ROOT / "requirements-colab.txt"


if IN_COLAB and INSTALL_REQUIREMENTS_IN_COLAB and requirements_path.exists() and missing_imports:
    print(f"Installing Colab requirements from {requirements_path}.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)])
else:
    for import_name, pip_name in REQUIRED_NOTEBOOK_PACKAGES:
        ensure_notebook_package(import_name, pip_name)

Custom Modules and Libraries

In [ ]:
from modules.data_setup import (
    adapt_cuad_to_clause_classification,
    build_project_paths,
    download_cuad_if_missing,
    load_cuad_raw_files,
    load_or_download_ledgar,
    print_dataset_availability,
    seed_everything,
)
from modules.preprocessing import create_ledgar_eda, preprocess_ledgar
from modules.baselines import run_baseline_experiments
from modules.classical_models import run_classical_experiments
from modules.transformer_model import train_transformer_classifier
from modules.qwen_prompting import run_qwen_baseline
from modules.agentic_review import run_agentic_review
from modules.evaluation import save_final_comparison
from modules.error_analysis import run_error_analysis
from modules.wandb_reporting import finish_wandb_run, log_wandb_outputs, start_wandb_run



| Setting | Value | Purpose |
|---|---:|---|
| `SEED` | `42` | Makes sampling, baseline randomness, and train/test helper behavior reproducible. |
| `DATASET_NAME` | `LEDGAR` | Keeps the main experiment scoped to LEDGAR clause classification. |
| `TOP_K_LABELS` | `20` | Restricts the task to the 20 most frequent training labels for a manageable coursework experiment. |
| `RUN_CLASSICAL_MODELS` | `True` | Enables TF-IDF model experiments. |
| `RUN_TRANSFORMER` | `True` | Attempts transformer fine-tuning only when the runtime can support it. |
| `RUN_QWEN_BASELINE` | `True` | Attempts Qwen prompting only when GPU/model loading is available. |
| `RUN_AGENTIC_EXTENSION` | `True` | Enables a small review workflow demonstration, not an autonomous agent. |
| `RUN_WANDB` | `True` | Sends metrics and safe artifacts to W&B when credentials are available. |

Model and feature hyperparameters declared here:

| Component | Hyperparameters |
|---|---|
| TF-IDF search | `max_features` in `[10000, 30000]`; `ngram_range` in `[(1, 1), (1, 2)]`; `lowercase=True`; `stop_words=None` |
| Transformer | `distilbert-base-uncased`; `max_length=256` |
| Optional legal transformer | `nlpaueb/legal-bert-base-uncased` can be substituted manually if GPU resources allow |
| Qwen prompting | `Qwen/Qwen2.5-3B-Instruct`; test sample size `200`; one few-shot example per class when available |
| W&B logging | Uses `WANDB_API_KEY` from Colab Secrets or the environment; text-containing prediction/error tables are not uploaded unless `WANDB_LOG_TEXT_TABLES=True` |

Explainability note: keeping all configuration values in one cell makes it clear which choices affect runtime cost, model capacity, and evaluation scope.

In [ ]:
SEED = 42

DATASET_NAME = "LEDGAR"

TOP_K_LABELS = 20

MAX_FEATURES_LIST = [10000, 30000]

NGRAM_RANGES = [(1, 1), (1, 2)]

RUN_CLASSICAL_MODELS = True

RUN_TRANSFORMER = True

RUN_QWEN_BASELINE = True

RUN_AGENTIC_EXTENSION = True

RUN_NAIVE_BAYES = True

RUN_WANDB = True

WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "ledgar-clause-classification")

WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "").strip() or None

WANDB_MODE = os.environ.get("WANDB_MODE", "online")

WANDB_LOG_ARTIFACTS = True

WANDB_LOG_TEXT_TABLES = False

WANDB_LOG_MODEL_FILES = False

TRANSFORMER_MODEL_NAME = "distilbert-base-uncased"

OPTIONAL_LEGAL_MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

QWEN_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

MAX_TRANSFORMER_LENGTH = 256

QWEN_EVAL_SAMPLE_SIZE = 200

QWEN_FEW_SHOT_EXAMPLES_PER_CLASS = 1


DOWNLOAD_LEDGAR_IF_MISSING = True

DOWNLOAD_CUAD_IF_MISSING = True

USE_HF_CACHE = True

FORCE_REDOWNLOAD = False

paths = build_project_paths(PROJECT_ROOT)
DEVICE = seed_everything(SEED)


Weights and Biases Setup

In [ ]:
print(f"Project root: {paths.project_root}")
print(f"Colab runtime: {IN_COLAB}")
print(f"Raw LEDGAR directory: {paths.ledgar_raw_dir}")
print(f"Results directory: {paths.results_dir}")
print(f"Device: {DEVICE}")

In [ ]:

try:
    import torch

    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("GPU is unavailable. Transformer/Qwen sections will skip or reduce work gracefully.")
except Exception:
    print("PyTorch is unavailable. Transformer/Qwen sections will skip if they require it.")

wandb_run = start_wandb_run(
    enabled=RUN_WANDB,
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    group="ledgar-coursework",
    tags=["ledgar", "legal-clause-classification", "coursework"],
    config={
        "seed": SEED,
        "dataset_name": DATASET_NAME,
        "top_k_labels": TOP_K_LABELS,
        "run_classical_models": RUN_CLASSICAL_MODELS,
        "run_transformer": RUN_TRANSFORMER,
        "run_qwen_baseline": RUN_QWEN_BASELINE,
        "run_agentic_extension": RUN_AGENTIC_EXTENSION,
        "run_naive_bayes": RUN_NAIVE_BAYES,
        "transformer_model_name": TRANSFORMER_MODEL_NAME,
        "max_transformer_length": MAX_TRANSFORMER_LENGTH,
        "qwen_model_name": QWEN_MODEL_NAME,
        "qwen_eval_sample_size": QWEN_EVAL_SAMPLE_SIZE,
        "device": str(DEVICE),
        "log_text_tables": WANDB_LOG_TEXT_TABLES,
        "log_model_files": WANDB_LOG_MODEL_FILES,
    },
    mode=WANDB_MODE,
)

WANDB_ACTIVE = wandb_run is not None


## 2. Dataset Download and Raw Setup

This stage obtains the raw datasets without training or preprocessing models. LEDGAR remains the main classification dataset. CUAD is treated separately because it is structured as a contract-review question-answering/span-extraction dataset rather than a direct clause-classification dataset.

Data governance choices:

- LEDGAR is loaded from Hugging Face with `load_dataset("coastalcph/lex_glue", "ledgar")` when local JSONL files are missing.
- Official LEDGAR train, validation, and test splits are preserved when available.
- Raw LEDGAR split exports are saved under `data/raw/lexglue_ledgar/` as JSONL files.
- CUAD raw files are downloaded from `theatticusproject/cuad` when available, but CUAD is not merged with LEDGAR.
- If CUAD is missing, the notebook prints a clear message and continues with LEDGAR.

Inputs and outputs:

| Input | Output |
|---|---|
| Hugging Face LEDGAR or local JSONL | `ledgar_raw_splits` dictionary with train/validation/test DataFrames |
| Optional CUAD raw files | `cuad_clause_df` containing extracted span-level examples for optional inspection |

This stage deliberately does not select labels, encode classes, train models, or compute metrics.


CAUD Dataset

In [ ]:
ledgar_raw_splits = load_or_download_ledgar(
    paths,
    download_if_missing=DOWNLOAD_LEDGAR_IF_MISSING,
    force_redownload=FORCE_REDOWNLOAD,
)

cuad_json_path, master_clauses_path = download_cuad_if_missing(
    paths,
    download_if_missing=DOWNLOAD_CUAD_IF_MISSING,
    force_redownload=FORCE_REDOWNLOAD,
)

raw_cuad_json, master_clauses_df = load_cuad_raw_files(cuad_json_path, master_clauses_path)
cuad_clause_df = adapt_cuad_to_clause_classification(raw_cuad_json)
print_dataset_availability(ledgar_raw_splits, cuad_json_path, master_clauses_path, cuad_clause_df)

Loading LEDGAR from data/raw/lexglue_ledgar JSONL files.
LEDGAR train: 60000 rows, columns=['text', 'label']
LEDGAR validation: 10000 rows, columns=['text', 'label']
LEDGAR test: 10000 rows, columns=['text', 'label']
Using existing CUAD files from data/raw/cuad/.

Dataset availability:
- LEDGAR downloaded/loaded: yes
- LEDGAR train size: 60000
- LEDGAR validation size: 10000
- LEDGAR test size: 10000
- CUAD JSON found: yes
- CUAD master clauses CSV found: yes
- CUAD adapted span examples available for optional analysis: 13062


## 3. LEDGAR Preprocessing and EDA

This stage converts raw LEDGAR into a consistent clause-classification schema used by all later experiments.

Preprocessing decisions:

- Standard schema: `text`, `label`, `label_id`, `split`, `source_dataset`.
- Text cleaning is intentionally light: whitespace is normalised and leading/trailing spaces are stripped.
- Legal punctuation and stopwords are retained because they may carry meaning in contractual language.
- Empty or malformed examples are removed.
- Exact duplicate `text` plus `label` pairs are removed to reduce repeated rows.
- The top `TOP_K_LABELS=20` labels are selected using the training split only, preventing validation/test leakage in label selection.
- Label IDs are assigned after filtering so every model uses the same `label2id` and `id2label` mappings.

EDA outputs generated here:

| Output | Purpose |
|---|---|
| `class_distribution.png` | Shows imbalance across the selected labels. |
| `clause_length_histogram.png` | Shows clause length variation, useful for interpreting transformer truncation risk. |
| `dataset_split_summary.csv` | Records split sizes and class counts. |
| `examples_per_label.jsonl` | Provides qualitative examples for annotation and ambiguity inspection. |

The processed JSONL files are saved under `data/processed/` and become the controlled inputs for all model sections.


In [ ]:
processed_splits, label2id, id2label = preprocess_ledgar(

    ledgar_raw_splits,

    paths,

    top_k_labels=TOP_K_LABELS,

    dataset_name=DATASET_NAME,
)

split_summary = create_ledgar_eda(processed_splits, paths.results_dir)


if processed_splits:
    train_df = processed_splits["train"]
    validation_df = processed_splits["validation"]
    test_df = processed_splits["test"]
    label_names = [id2label[i] for i in sorted(id2label)]
    display(split_summary)
    display(pd.DataFrame({"label": label_names}))

else:
    train_df = validation_df = test_df = pd.DataFrame(columns=["text", 
                                                               "label", 
                                                               "label_id", 
                                                               "split", 
                                                               "source_dataset"])
    
    label_names = []

    
    print("Main LEDGAR experiment cannot run without LEDGAR data.")

,split,rows,classes
0,train,28587,20
1,validation,4670,20
2,test,4732,20


,label
0,Governing Laws
1,Notices
2,Counterparts
3,Entire Agreements
4,Severability
5,Amendments
6,Survival
7,Assignments
8,Expenses
9,Terms


## 4. Shared Result State

This short stage creates shared containers used by the later model sections.

- `completed_results` stores one row per completed or skipped model run.
- `prediction_tables` stores per-example predictions for error analysis.
- `trained_models` stores reusable fitted model objects when available.

Governance purpose: every model section appends to the same result structure, so the final comparison table is generated from actual run outputs rather than manually entered values.


Variable Initialization

In [ ]:
completed_results = []

prediction_tables = {}

trained_models = {}

## 5. Dummy Baselines

This stage evaluates non-learning baselines. These baselines are important because they establish a minimum reference point before interpreting more complex models.

Baselines used:

| Model | Behavior | Why it matters |
|---|---|---|
| `random_uniform` | Samples uniformly from the selected label IDs. | Tests performance expected from chance under equal class probability. |
| `random_train_distribution` | Samples labels according to the training label distribution. | Reflects class imbalance without learning from text. |
| `majority_baseline` | Always predicts the most frequent training label. | Provides a strong imbalance-aware dummy baseline for accuracy comparison. |

Evaluation metrics saved for each baseline:

- accuracy
- macro-F1
- weighted-F1
- per-class precision/recall/F1 via classification report
- confusion matrix

Macro-F1 is the primary governance metric because it penalises poor performance on minority classes more clearly than accuracy.


In [ ]:
baseline_results, baseline_prediction_tables = run_baseline_experiments(
    
    train_df,
    
    test_df,
    
    id2label,
    
    paths.results_dir,
    
    dataset_name=DATASET_NAME,
    
    seed=SEED,
)


completed_results.extend(baseline_results)

prediction_tables.update(baseline_prediction_tables)

if baseline_results:
    display(pd.DataFrame(baseline_results)[["model_name", "accuracy", "macro_f1", "weighted_f1", "notes"]])

,model_name,accuracy,macro_f1,weighted_f1,notes
0,random_uniform,0.049451,0.045640,0.052960,Uniform random over selected labels.
1,random_train_distribution,0.059383,0.044516,0.060032,Random predictions sampled from the training l...
2,majority_baseline,0.120034,0.010717,0.025728,Always predicts the most frequent training label.


## 6. Classical TF-IDF Models

This stage trains sparse-text supervised models using TF-IDF features. These models are fast, interpretable at the feature level, and provide strong non-neural baselines for legal text classification.


Selection protocol:

1. Train each configuration on the LEDGAR training split.
2. Select the best configuration using validation macro-F1.
3. Evaluate selected models on the test split once.
4. Save the best classical pipeline and vectorizer artifacts separately.

Explainability note: TF-IDF models are useful for coursework governance because their decisions are linked to sparse lexical features rather than hidden contextual embeddings.


Variable Initialization

In [ ]:
# Initialize variables to track the best classical model and its name, which will be updated after running classical experiments.

best_classical_model = None

best_classical_name = None

Training for Classical Machine-Learning Models

Feature extraction hyperparameters:

| Hyperparameter | Values |
|---|---|
| `max_features` | `10000`, `30000` |
| `ngram_range` | unigram `(1, 1)`, unigram+bigram `(1, 2)` |
| `lowercase` | `True` |
| `stop_words` | `None` |

Model configurations:

| Model | Key settings |
|---|---|
| Logistic Regression | `max_iter=2000`; `random_state=42`; tests `class_weight=None` and `class_weight="balanced"` |
| Linear SVM | `LinearSVC`; `random_state=42`; tests `class_weight=None` and `class_weight="balanced"` |
| Multinomial Naive Bayes | Optional comparison model using TF-IDF inputs |


In [ ]:

# Training with Manual Hyperparameter Tuning

if not RUN_CLASSICAL_MODELS:
    print("RUN_CLASSICAL_MODELS doesn't exist or is set to False.")

else:

    classical_output = run_classical_experiments(

        train_df, 
        # Training data for classical models

        validation_df, 
        # Validation data for classical models (used for hyperparameter tuning)

        test_df, 
        # Test data for classical models

        id2label, 
        # Mapping from label IDs to label names

        paths.results_dir, 
        # Directory to save results and artifacts

        max_features_list=MAX_FEATURES_LIST, 
        # List of max_features values to try for CountVectorizer/TfidfVectorizer

        ngram_ranges=NGRAM_RANGES, 
        # List of ngram_range tuples to try for CountVectorizer/TfidfVectorizer

        dataset_name=DATASET_NAME, 
        # Name of the dataset (used for logging and artifact naming)

        seed=SEED, 
        # Random seed for reproducibility

        run_naive_bayes=RUN_NAIVE_BAYES, 
        # Whether to run Naive Bayes models (MultinomialNB, ComplementNB)
    )


# Aggregration of classical model results and prediction tables into the overall results and prediction tables.

    completed_results.extend(classical_output["results"]) 
    prediction_tables.update(classical_output["prediction_tables"]) 

# Best Model Selection and Display

    best_classical_model = classical_output["best_model"] 
    best_classical_name = classical_output["best_model_name"] 



    if best_classical_model is not None:

        trained_models["best_classical"] = best_classical_model

# Print classical machine-Learning Models for 
# "Model Name", 
# "Accuracy", 
# "Macro F1", 
# "Weighted F1",
# "Notes" columns 

# Only print if there are results to show, otherwise skip to avoid empty table display.

    if classical_output["results"]:
        
        display(pd.DataFrame(classical_output["results"])[["model_name", "accuracy", "macro_f1", "weighted_f1", "notes"]])

,model_name,accuracy,macro_f1,weighted_f1,notes
0,logistic_regression,0.954987,0.941931,0.954893,Selected on validation macro-F1. Config={'mode...
1,linear_svm,0.963863,0.953366,0.963473,Selected on validation macro-F1. Config={'mode...
2,multinomial_nb,0.932587,0.911258,0.930669,Selected on validation macro-F1. Config={'mode...


## 7. Fine-Tuned Transformer Classifier

This stage optionally fine-tunes a Hugging Face sequence-classification transformer on LEDGAR. The default model is `distilbert-base-uncased` because it is smaller and more practical for coursework hardware than full BERT-size alternatives.

Training settings used by the module:

| Setting | Value |
|---|---:|
| Model | `distilbert-base-uncased` |
| Maximum sequence length | `256` tokens |
| Learning rate | `2e-5` |
| Epochs | `3` |
| Weight decay | `0.01` |
| Batch size | `16` on larger GPUs, otherwise `8` |
| Mixed precision | `fp16=True` when CUDA is available |
| Model selection | best validation `macro_f1` |
| Early stopping | patience `1` when the callback is available |

Runtime governance:

- This section skips gracefully if CUDA/GPU is unavailable.
- Memory or environment failures are caught and recorded as skipped results.
- The transformer is evaluated on the same LEDGAR test labels as the classical models.

Explainability limitation: transformer representations are contextual but less directly inspectable than TF-IDF features, so confusion matrices and misclassified examples are important for interpreting behavior.


In [ ]:
transformer_output = train_transformer_classifier(

    train_df,
    # Training data for transformer model

    validation_df,
    # Validation data for transformer model (used for early stopping and hyperparameter tuning)
    
    test_df,
    # Test data for transformer model

    id2label,
    # Mapping from label IDs to label names

    paths.results_dir,
    # Directory to save results and artifacts

    model_name=TRANSFORMER_MODEL_NAME,
    # Name of the transformer model to use (e.g., "distilbert-base-uncased")

    max_length=MAX_TRANSFORMER_LENGTH,
    # Maximum sequence length for transformer inputs

    dataset_name=DATASET_NAME,
    # Name of the dataset (used for logging and artifact naming)

    seed=SEED,
    # Random seed for reproducibility

    run_transformer=RUN_TRANSFORMER,
    # Whether to run the transformer model training and evaluation

    wandb_enabled=WANDB_ACTIVE,
    # Whether Weights & Biases logging is enabled, passed to the training function for logging purposes

    wandb_run_name=getattr(wandb_run, "name", None),
    # Name of the Weights & Biases run, passed to the training function for logging purposes
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,model_name,accuracy,macro_f1,weighted_f1
0,distilbert-base-uncased,0.96492,0.954658,0.964788


In [ ]:

# If transformer_output contains a valid output, 
#
#   append the results to completed_results,
#   update the prediction_tables with the transformer's predictions,
#   store the trained transformer model in trained_models under the key "transformer_trainer",
#   and display a DataFrame with the transformer's results showing:
#
#       "model_name", 
#       "accuracy",
#       "macro_f1",
#       "weighted_f1" columns.

if transformer_output["result"] is not None:
    completed_results.append(transformer_output["result"])
    prediction_tables[TRANSFORMER_MODEL_NAME] = transformer_output["predictions"]
    trained_models["transformer_trainer"] = transformer_output["trainer"]
    display(pd.DataFrame([transformer_output["result"]])[["model_name", 
                                                          "accuracy", 
                                                          "macro_f1", 
                                                          "weighted_f1"]])


elif transformer_output["skip_result"] is not None:
    completed_results.append(transformer_output["skip_result"])

## 8. Qwen2.5-Instruct Prompting Baseline

This stage optionally evaluates an instruction-tuned language model as a prompting baseline. Qwen is not fine-tuned; it is only prompted to classify clauses into the fixed LEDGAR label set.

Prompting setup:

| Mode | Description |
|---|---|
| Zero-shot | Provides the clause text and full list of allowed labels. |
| Few-shot | Adds one training example per label when available; validation and test examples are never used as demonstrations. |

Generation and parsing controls:

| Setting | Value |
|---|---:|
| Model | `Qwen/Qwen2.5-3B-Instruct` |
| Evaluation sample | up to `200` test examples, sampled with `SEED=42` |
| Decoding | deterministic, `do_sample=False` |
| New tokens | `max_new_tokens=24` |
| Output requirement | return exactly one allowed label |
| Fuzzy matching | only used when unambiguous, cutoff `0.80` |
| Invalid outputs | marked as `INVALID_PREDICTION` and reported separately |

Governance note: this is not directly equivalent to supervised fine-tuning. The prompted model has different pretraining and task setup, so results should be interpreted as a separate baseline rather than a perfectly fair model-family comparison.


In [ ]:


qwen_output = run_qwen_baseline(
   
    train_df, 
    # Training data for Qwen prompting (used to create few-shot examples)

    test_df,
    # Test data for Qwen prompting (used for evaluation)
    
    label2id,
    # Mapping from label names to label IDs, which may be needed for formatting prompts or interpreting outputs
   
    id2label,
    # Mapping from label IDs to label names, which may be needed for formatting prompts or interpreting outputs
   
    paths.results_dir,
    # Directory to save results and artifacts related to the Qwen baseline
   
    model_name=QWEN_MODEL_NAME,
    # Name of the Qwen model to use for prompting (e.g., "Qwen/Qwen2.5-3B-Instruct")
   
    label_names=label_names,
    # List of label names corresponding to the label IDs, which may be needed for formatting prompts or interpreting outputs
   
    eval_sample_size=QWEN_EVAL_SAMPLE_SIZE,
    # Number of test samples to evaluate on for the Qwen baseline (e.g., 200)
   
    few_shot_examples_per_class=QWEN_FEW_SHOT_EXAMPLES_PER_CLASS,
    # Number of few-shot examples to include per class in the prompt for Qwen (e.g., 1)
   
    dataset_name=DATASET_NAME,
    # Name of the dataset (used for logging and artifact naming)

    seed=SEED,
    # Random seed for reproducibility, which may be used for sampling evaluation examples or shuffling data

    run_qwen=RUN_QWEN_BASELINE,
    # Whether to run the Qwen baseline (if False, the function may skip execution and return None or a skip result)
)



config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,model_name,accuracy,macro_f1,weighted_f1,notes
0,qwen_zero_shot,0.565,0.503949,0.568809,Qwen prompting baseline. Invalid prediction ra...
1,qwen_few_shot,0.110,0.009910,0.021802,Qwen prompting baseline. Invalid prediction ra...


In [ ]:

completed_results.extend(qwen_output["results"])

qwen_predictions_df = qwen_output["predictions"]

qwen_invalid_outputs_df = qwen_output["invalid_outputs"]

qwen_model = qwen_output["model"]

qwen_tokenizer = qwen_output["tokenizer"]

if qwen_output["results"]:
    display(pd.DataFrame(qwen_output["results"])[[
        "model_name", 
        "accuracy", 
        "macro_f1", 
        "weighted_f1", 
        "notes"]])

#
# If qwen_output contains valid results,                                                    
#   append the results to completed_results,                                                     
#   store the predictions, invalid outputs, model, and tokenizer in respective variables,       
#   and display a DataFrame with the Qwen baseline results showing:                          
#       "model_name",                                                                             
#       "accuracy",                                                                                
#       "macro_f1",                                                                                
#       "weighted_f1",                                                                             
#       "notes" columns.                                                   
#

## 9. Small Agentic Review Prototype

This stage demonstrates a small human-in-the-loop clause triage workflow. It is inspired by tool-use/ReAct-style ideas, but it is not a large autonomous agent and it does not provide legal advice.

Workflow:

1. Use the best available supervised classifier to predict a clause type.
2. Estimate prediction confidence where the model supports it.
3. Flag examples below the review threshold as requiring human review.
4. Optionally ask Qwen for a short triage explanation when Qwen loaded successfully.

Prototype settings:

| Setting | Value |
|---|---:|
| Sample size | `20` test examples |
| Review threshold | `0.55` confidence |
| Logistic Regression confidence | maximum predicted probability |
| Linear SVM confidence | transformed decision-function margin |
| Required disclaimer | `This output is for clause triage and research purposes only.` |

Governance limitation: this section is illustrative. It should not be treated as a production legal review system, risk model, or legal advisor.


In [ ]:
agentic_examples_df = run_agentic_review(

    test_df,
    # Test data for agentic review (used to identify examples for review and potential correction)
    
    id2label,
    # Mapping from label IDs to label names, which may be needed for formatting prompts or interpreting outputs

    paths.results_dir,

    # Directory to save results and artifacts related to the agentic review
    best_model=best_classical_model,

    # The best classical model identified from the classical experiments, which may be used as a baseline for comparison or to identify examples where it fails
    qwen_model=qwen_model,

    # The Qwen model used for prompting, which may be used to generate explanations or corrections for misclassified examples
    qwen_tokenizer=qwen_tokenizer,
    
    # Name of the dataset (used for logging and artifact naming)
    run_agentic=RUN_AGENTIC_EXTENSION,

    # Random seed for reproducibility, which may be used for sampling examples for review or shuffling data
    seed=SEED,

    # Whether to run the agentic review process (if False, the function may skip execution and return an empty DataFrame or None
)

if not agentic_examples_df.empty:
    display(agentic_examples_df.head(10))

,text,true_label,predicted_label,confidence,requires_human_review,triage_note,optional_qwen_explanation
0,"Each party hereto shall do and perform, or cau...",Further Assurances,Further Assurances,0.760225,False,This output is for clause triage and research ...,
1,"There is no pending or threatened notice, clai...",Litigations,Litigations,0.724625,False,This output is for clause triage and research ...,
2,The term of this Agreement shall commence on J...,Terms,Terms,0.697402,False,This output is for clause triage and research ...,
3,"The Executive acknowledges that, by reason of ...",Assignments,Assignments,0.648360,False,This output is for clause triage and research ...,
4,"Pledgor will, from time to time, timely pay an...",Taxes,Taxes,0.694286,False,This output is for clause triage and research ...,
5,Except as set forth on Schedule 3.6 as of the ...,Litigations,Litigations,0.663922,False,This output is for clause triage and research ...,
6,This Warrant shall be governed by and construe...,Governing Laws,Governing Laws,0.675921,False,This output is for clause triage and research ...,
7,This Assignment constitutes the entire and fin...,Entire Agreements,Entire Agreements,0.679556,False,This output is for clause triage and research ...,
8,"The term of this Sublease (""Term"") shall comme...",Terms,Terms,0.565141,False,This output is for clause triage and research ...,
9,This Amendment may be executed by the parties ...,Counterparts,Counterparts,0.719196,False,This output is for clause triage and research ...,


## 10. Final Model Comparison

This stage consolidates all completed and skipped model runs into one comparison table. It does not insert or fabricate any metrics; it only formats rows produced by earlier sections.

Comparison columns:

| Column | Meaning |
|---|---|
| `model_family` | baseline, classical, transformer, or prompting family. |
| `model_name` | specific model/configuration name. |
| `training_type` | dummy, supervised, fine-tuned, prompted, or skipped. |
| `dataset` | evaluation dataset, here LEDGAR for the main experiment. |
| `eval_split` | split used for reported metrics, usually test. |
| `sample_size` | number of evaluated examples. |
| `accuracy` | overall exact-label accuracy. |
| `macro_f1` | unweighted mean F1 across classes; primary metric. |
| `weighted_f1` | class-frequency-weighted F1. |
| `notes` | skip reason or relevant run detail. |

The table and macro-F1 plot are saved under `results/` for later inspection.


In [ ]:
comparison_df = save_final_comparison(completed_results, paths.results_dir)

print(f"Saved final comparison to: {paths.results_dir / 'final_model_comparison.csv'}")

display(comparison_df)

Saved final comparison to: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/results/final_model_comparison.csv


,model_family,model_name,training_type,dataset,eval_split,sample_size,accuracy,macro_f1,weighted_f1,invalid_prediction_rate,notes,classification_report_path,confusion_matrix_path
0,baseline,random_uniform,dummy,LEDGAR,test,4732,0.049451,0.045640,0.052960,NaN,Uniform random over selected labels.,/content/drive/MyDrive/Colab Notebooks/Educati...,/content/drive/MyDrive/Colab Notebooks/Educati...
1,baseline,random_train_distribution,dummy,LEDGAR,test,4732,0.059383,0.044516,0.060032,NaN,Random predictions sampled from the training l...,/content/drive/MyDrive/Colab Notebooks/Educati...,/content/drive/MyDrive/Colab Notebooks/Educati...
2,baseline,majority_baseline,dummy,LEDGAR,test,4732,0.120034,0.010717,0.025728,NaN,Always predicts the most frequent training label.,/content/drive/MyDrive/Colab Notebooks/Educati...,/content/drive/MyDrive/Colab Notebooks/Educati...
3,classical,logistic_regression,supervised,LEDGAR,test,4732,0.954987,0.941931,0.954893,NaN,Selected on validation macro-F1. Config={'mode...,/content/drive/MyDrive/Colab Notebooks/Educati...,/content/drive/MyDrive/Colab Notebooks/Educati...
4,classical,linear_svm,supervised,LEDGAR,test,4732,0.963863,0.953366,0.963473,NaN,Selected on validation macro-F1. Config={'mode...,/content/drive/MyDrive/Colab Notebooks/Educati...,/content/drive/MyDrive/Colab Notebooks/Educati...
5,classical,multinomial_nb,supervised,LEDGAR,test,4732,0.932587,0.911258,0.930669,NaN,Selected on validation macro-F1. Config={'mode...,/content/drive/MyDrive/Colab Notebooks/Educati...,/content/drive/MyDrive/Colab Notebooks/Educati...
6,transformer,distilbert-base-uncased,fine-tuned supervised,LEDGAR,test,4732,0.964920,0.954658,0.964788,NaN,Fine-tuned Hugging Face sequence classifier.,/content/drive/MyDrive/Colab Notebooks/Educati...,/content/drive/MyDrive/Colab Notebooks/Educati...
7,llm_prompting,qwen_zero_shot,zero_shot,LEDGAR,test,200,0.565000,0.503949,0.568809,0.045,Qwen prompting baseline. Invalid prediction ra...,/content/drive/MyDrive/Colab Notebooks/Educati...,/content/drive/MyDrive/Colab Notebooks/Educati...
8,llm_prompting,qwen_few_shot,few_shot,LEDGAR,test,200,0.110000,0.009910,0.021802,1.000,Qwen prompting baseline. Invalid prediction ra...,/content/drive/MyDrive/Colab Notebooks/Educati...,/content/drive/MyDrive/Colab Notebooks/Educati...


## 11. Error Analysis

This stage checks model behaviour beyond aggregate metrics. It uses saved predictions, confusion outputs, and class counts from earlier stages.

Outputs reviewed:

| Output | Purpose |
|---|---|
| Top confused label pairs | Shows where the best classical model mixes labels. |
| Misclassified examples | Provides examples for qualitative inspection. |
| Transformer errors | Included only when the transformer ran. |
| Qwen invalid outputs | Included only when Qwen produced real predictions. |
| Class imbalance summary | Shows how training labels are distributed. |

These outputs support later discussion without writing report conclusions here.


In [ ]:
error_outputs = run_error_analysis(
    comparison_df,
    prediction_tables,
    train_df,
    paths.results_dir,
    best_classical_name=best_classical_name,
    transformer_model_name=TRANSFORMER_MODEL_NAME,
    qwen_predictions_df=qwen_predictions_df,
    qwen_invalid_outputs_df=qwen_invalid_outputs_df,
)


Best completed model by macro-F1: distilbert-base-uncased
Top classical confused label pairs:


,label,predicted_label,count
76,Terms,General,9
36,General,Terminations,8
35,General,Taxes,6
25,General,Assignments,6
37,General,Terms,6
54,Notices,General,5
71,Terminations,Terms,5
1,Amendments,Entire Agreements,5
38,General,Waivers,4
79,Terms,Terminations,4


Classical misclassified examples:


,text,label,predicted_label
38,Upon consummation of an IPO or a Sale of the C...,Terminations,Survival
47,Except as otherwise provided below or in the A...,General,Terms
91,Any Property Costs that are not reflected in t...,Expenses,Taxes
108,Perry acknowledges that the terms of this Agre...,Compliance With Laws,Waivers
115,Subject to the provisions set forth in Article...,General,Indemnifications
182,Except as shall otherwise be stated herein or ...,General,Terms
200,Each Subsidiary of the Company that is or beco...,Notices,General
228,Upon the Completion Date with respect to the F...,Further Assurances,Terminations
240,"The costs and expenses (including, but not lim...",Expenses,Indemnifications
356,Notices and all other communications contempla...,General,Notices


Transformer misclassified examples:


...

,mode,text,label,label_id,raw_output,predicted_label,predicted_label_id,is_invalid
32,zero_shot,The Award of Performance Shares (as set forth ...,Terms,9,Governing Laws\nExplanation: The clause speci...,INVALID_PREDICTION,<NA>,True
42,zero_shot,"Each Party shall deliver all notices, requests...",Notices,1,"Notions\nThe predicted label ""Notions"" does n...",INVALID_PREDICTION,<NA>,True
60,zero_shot,"All demands, notices and communications hereun...",Notices,1,"Notions\nThe predicted label ""Notions"" does n...",INVALID_PREDICTION,<NA>,True
72,zero_shot,Any notice or document required to be filed wi...,Notices,1,General\nThe predicted label for the given cl...,INVALID_PREDICTION,<NA>,True
86,zero_shot,"The Borrower will, and will cause each Princip...",Taxes,12,Compliance With Laws\nExplanation: The clause...,INVALID_PREDICTION,<NA>,True
171,zero_shot,The Guarantor shall pay on demand all out-of-p...,Expenses,8,Compliance With Laws\nThis prediction seems i...,INVALID_PREDICTION,<NA>,True
174,zero_shot,Any notice to the Company provided for in this...,Notices,1,"Notions\nThe predicted label ""Notions"" does n...",INVALID_PREDICTION,<NA>,True
198,zero_shot,The Consolidated Parties have filed all Federa...,Taxes,12,Compliance With Laws\nExplanation: The clause...,INVALID_PREDICTION,<NA>,True
199,zero_shot,"This Confirmation is effective as of April 1, ...",Terms,9,Governing Laws\nThe predicted label does not ...,INVALID_PREDICTION,<NA>,True
200,few_shot,"Each party hereto shall do and perform, or cau...",Further Assurances,14,"such actions, as may be necessary or desirabl...",INVALID_PREDICTION,<NA>,True


Qwen semantically plausible but non-matching label examples require manual inspection:


,mode,text,label,label_id,raw_output,predicted_label,predicted_label_id,is_invalid
1,zero_shot,"There is no pending or threatened notice, clai...",Litigations,13,Compliance With Laws,Compliance With Laws,17,False
2,zero_shot,The term of this Agreement shall commence on J...,Terms,9,Compliance With Laws,Compliance With Laws,17,False
3,zero_shot,"The Executive acknowledges that, by reason of ...",Assignments,7,Compliance With Laws\nThis clause discusses o...,Compliance With Laws,17,False
5,zero_shot,Except as set forth on Schedule 3.6 as of the ...,Litigations,13,Compliance With Laws,Compliance With Laws,17,False
7,zero_shot,This Assignment constitutes the entire and fin...,Entire Agreements,3,Amendments,Amendments,5,False
12,zero_shot,"Borrower shall, and shall cause each Credit Pa...",Insurances,11,Compliance With Laws,Compliance With Laws,17,False
14,zero_shot,To request the issuance of a Letter of Credit ...,Amendments,5,Compliance With Laws,Compliance With Laws,17,False
19,zero_shot,"Subject to the other terms of this Agreement, ...",Further Assurances,14,General,General,16,False
20,zero_shot,Each Credit Party executing this Agreement ack...,Terminations,10,Compliance With Laws,Compliance With Laws,17,False
23,zero_shot,"As between the First Lien Secured Parties, the...",Insurances,11,Compliance With Laws,Compliance With Laws,17,False


Class imbalance summary:


,label,train_count
0,Governing Laws,3136
1,Notices,2445
2,Counterparts,2376
3,Entire Agreements,2318
4,Severability,1774
5,Amendments,1460
6,Survival,1442
7,Assignments,1308
8,Expenses,1211
9,Terms,1142


Interpretation focus:

| Issue | Why it matters |
|---|---|
| Class imbalance | Accuracy can look high while minority classes perform poorly. |
| Label ambiguity | Legal clauses may plausibly fit more than one clause type. |
| Long clauses | Transformer truncation and TF-IDF sparsity can affect predictions. |
| Boilerplate wording | Repeated legal phrasing can make labels harder to separate. |
| Invalid LLM outputs | Prompted models may ignore the closed label set. |

In [ ]:
print(f"Best completed model by macro-F1: {error_outputs.get('best_model_name')}")

In [ ]:
if "classical_confusions" in error_outputs:
    print("Top classical confused label pairs:")
    display(error_outputs["classical_confusions"])

if "classical_misclassified" in error_outputs:
    print("Classical misclassified examples:")
    display(error_outputs["classical_misclassified"][["text", "label", "predicted_label"]])

if "transformer_misclassified" in error_outputs:
    print("Transformer misclassified examples:")
    display(error_outputs["transformer_misclassified"][["text", "label", "predicted_label"]])

if "qwen_invalid_outputs" in error_outputs:
    print("Qwen invalid outputs:")
    display(error_outputs["qwen_invalid_outputs"].head(10))

if "qwen_plausible_nonmatching" in error_outputs:
    print("Qwen plausible non-matching label examples for manual inspection:")
    display(error_outputs["qwen_plausible_nonmatching"].head(10))

print("Class imbalance summary:")
display(error_outputs.get("class_imbalance", pd.DataFrame()).head(20))


## 12. Report Artifact Exports

This stage writes the report-facing tables, figures, prompt outputs, and environment metadata. It does not edit `report.tex` or create metrics that were not produced earlier.

Key exports:

| Export type | Destination |
|---|---|
| CSV report tables | `outputs/` |
| Figures | `figures/` and `outputs/figures/` |
| Predictions and prompts | `outputs/` and `outputs/predictions/` |
| Runtime metadata | `outputs/environment.json` |

Skipped transformer or Qwen runs remain marked as skipped in the exported evidence.


In [ ]:
from modules.report_exports import export_report_artifacts

In [ ]:
report_artifacts = export_report_artifacts(
    paths=paths,
    processed_splits=processed_splits,
    label2id=label2id,
    id2label=id2label,
    completed_results=completed_results,
    prediction_tables=prediction_tables,
    error_outputs=error_outputs,
    qwen_predictions_df=qwen_predictions_df,
    qwen_invalid_outputs_df=qwen_invalid_outputs_df,
    seed=SEED,
    dataset_name=DATASET_NAME,
    max_features_list=MAX_FEATURES_LIST,
    ngram_ranges=NGRAM_RANGES,
    transformer_model_name=TRANSFORMER_MODEL_NAME,
    max_transformer_length=MAX_TRANSFORMER_LENGTH,
    qwen_model_name=QWEN_MODEL_NAME,
    qwen_eval_sample_size=QWEN_EVAL_SAMPLE_SIZE,
    qwen_few_shot_examples_per_class=QWEN_FEW_SHOT_EXAMPLES_PER_CLASS,
    run_naive_bayes=RUN_NAIVE_BAYES,
)

print("Saved report artifacts:")
for artifact_name, artifact_path in report_artifacts.items():
    print(f"- {artifact_name}: {artifact_path}")

# Log safe W&B outputs according to the notebook settings.
wandb_log_status = log_wandb_outputs(
    wandb_run,
    paths=paths,
    comparison_df=comparison_df,
    log_artifacts=WANDB_LOG_ARTIFACTS,
    log_text_tables=WANDB_LOG_TEXT_TABLES,
    log_model_files=WANDB_LOG_MODEL_FILES,
)

print(f"W&B status: {wandb_log_status}")
finish_wandb_run(wandb_run)
wandb_run = None
WANDB_ACTIVE = False


Saved report artifacts:
- data_summary: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/outputs/data_summary.csv
- label_distribution: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/outputs/label_distribution.csv
- main_results: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/outputs/main_results.csv
- predictions: {'distilbert-base-uncased': PosixPath('/content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/outputs/predictions/distilbert_base_uncased_test_predictions.jsonl'), 'linear_svm': PosixPath('/content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/outputs/predictions/linear_svm_test_predictions.jsonl'), 'logistic_regression': PosixPath('/content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/outputs/predictions/logistic_regression_test_predictions.jsonl'), 'majority_baseline': PosixPath('/content/drive/My

## 13. Method Notes and Limitations

These notes record modelling choices for transparency. They are not final coursework conclusions.

| Area | Note |
|---|---|
| Aim | Compare dummy baselines, classical supervised models, optional transformer fine-tuning, and optional Qwen prompting for LEDGAR clause classification. |
| Dataset | LEDGAR is the main supervised dataset. CUAD is separate and not merged into LEDGAR. |
| Preprocessing | The pipeline normalises whitespace, removes empty rows and exact duplicate text-label pairs, selects top-k training labels, and preserves official splits. |
| Baselines | Random and majority models provide lower-bound references. |
| Classical models | TF-IDF Logistic Regression, Linear SVM, and optional Naive Bayes provide sparse-text baselines. |
| Transformer | DistilBERT or LegalBERT is fine-tuned only when the runtime supports it. |
| Qwen | Qwen2.5-Instruct is used as zero-shot/few-shot prompting, not training. |
| Agentic extension | The prototype flags low-confidence examples for human review and is not legal advice. |

Limitations to discuss after results are generated: class imbalance, label ambiguity, long-clause truncation, prompt sensitivity, and differences between supervised and prompted model setups.


In [ ]:
# Locate project root.
try:
    PROJECT_ROOT_CHECK = Path(paths.project_root)
except NameError:
    PROJECT_ROOT_CHECK = Path.cwd()

REPORT_TEX = Path(r"C:\Users\ybenj\Downloads\report.tex")

# In Colab, this path only exists if report.tex was uploaded or synced.
# Standard report-facing artifacts are checked either way.
print(f"Project root: {PROJECT_ROOT_CHECK}")
print(f"report.tex found: {REPORT_TEX.exists()} -> {REPORT_TEX}")

# Required report-facing tables and files.
required_files = [
    "outputs/data_summary.csv",
    "outputs/label_distribution.csv",
    "outputs/main_results.csv",
    "outputs/per_class_results.csv",
    "outputs/confusion_pairs.csv",
    "outputs/misclassified_examples.csv",
    "outputs/hyperparameters.csv",
    "outputs/environment.json",
    "outputs/report_artifact_manifest.json",
    "data/processed/dataset_summary.json",
    "data/processed/label_counts.json",
    "data/processed/label_names.txt",
    "data/processed/ledgar_train.jsonl",
    "data/processed/ledgar_validation.jsonl",
    "data/processed/ledgar_test.jsonl",
]

# Conditional model evidence may contain skipped-status rows.
optional_but_expected_files = [
    "outputs/transformer_results.csv",
    "outputs/transformer_predictions.csv",
    "outputs/qwen_results.csv",
    "outputs/qwen_predictions.csv",
    "outputs/qwen_invalid_outputs.csv",
    "outputs/qwen_prompt_examples.txt",
    "results/transformer/runtime.json",
    "results/transformer/training_args.json",
    "results/transformer/training_log_history.json",
    "results/qwen/runtime.json",
    "results/qwen/qwen_run_config.json",
]

# Figures expected by the current report.
standard_report_figures = [
    "figures/label_distribution.png",
    "figures/clause_length_distribution.png",
    "figures/agentic_review_workflow.png",
    "figures/model_comparison_macro_f1.png",
    "figures/confusion_matrix_best_model.png",
    "figures/qwen_invalid_predictions.png",
]

# Add figure refs from report.tex when available.
figure_refs_from_tex = []
if REPORT_TEX.exists():
    tex = REPORT_TEX.read_text(encoding="utf-8", errors="ignore")
    figure_refs_from_tex = re.findall(r"\\includegraphics(?:\[[^\]]*\])?\{([^}]+)\}", tex)

figure_refs = sorted(set(standard_report_figures + figure_refs_from_tex))

def file_status(relative_path):
    path = PROJECT_ROOT_CHECK / relative_path
    exists = path.exists()
    size = path.stat().st_size if exists and path.is_file() else 0
    return {
        "path": relative_path,
        "exists": exists,
        "non_empty": bool(exists and size > 0),
        "size_bytes": size,
    }

def csv_rows(relative_path):
    path = PROJECT_ROOT_CHECK / relative_path
    if not path.exists() or path.stat().st_size == 0:
        return None
    try:
        return len(pd.read_csv(path))
    except Exception:
        return "unreadable"

def jsonl_rows(relative_path):
    path = PROJECT_ROOT_CHECK / relative_path
    if not path.exists() or path.stat().st_size == 0:
        return None
    return sum(1 for line in path.open("r", encoding="utf-8") if line.strip())

# Check files.
required_status = pd.DataFrame([file_status(p) for p in required_files])
optional_status = pd.DataFrame([file_status(p) for p in optional_but_expected_files])

required_status["rows_if_csv"] = required_status["path"].apply(lambda p: csv_rows(p) if p.endswith(".csv") else None)
optional_status["rows_if_csv"] = optional_status["path"].apply(lambda p: csv_rows(p) if p.endswith(".csv") else None)

print("\nREQUIRED FILES")
display(required_status)

print("\nOPTIONAL / CONDITIONAL MODEL EVIDENCE FILES")
display(optional_status)

# Check figures in report and output locations.
figure_rows = []
for ref in figure_refs:
    ref_path = Path(ref)
    candidates = [
        PROJECT_ROOT_CHECK / ref,
        PROJECT_ROOT_CHECK / "outputs" / ref,
    ]
    # Also check outputs/figures/foo.png for figures/foo.png refs.
    if len(ref_path.parts) >= 2 and ref_path.parts[0] == "figures":
        candidates.append(PROJECT_ROOT_CHECK / "outputs" / "figures" / ref_path.name)

    existing = [p for p in candidates if p.exists() and p.is_file() and p.stat().st_size > 0]
    figure_rows.append({
        "figure_ref": ref,
        "found": bool(existing),
        "found_at": str(existing[0]) if existing else "",
        "size_bytes": existing[0].stat().st_size if existing else 0,
    })

figure_status = pd.DataFrame(figure_rows)
print("\nFIGURES")
display(figure_status)

# Check prediction files against processed test size.
test_rows = jsonl_rows("data/processed/ledgar_test.jsonl")
prediction_dir = PROJECT_ROOT_CHECK / "outputs" / "predictions"
prediction_rows = []

if prediction_dir.exists():
    for pred_path in sorted(prediction_dir.glob("*_test_predictions.jsonl")):
        count = sum(1 for line in pred_path.open("r", encoding="utf-8") if line.strip())
        prediction_rows.append({
            "prediction_file": str(pred_path.relative_to(PROJECT_ROOT_CHECK)),
            "rows": count,
            "matches_test_rows": count == test_rows,
            "expected_test_rows": test_rows,
        })

prediction_status = pd.DataFrame(prediction_rows)
print("\nPREDICTION ROW COUNTS")
display(prediction_status)

# Final pass/fail summary.
missing_required = required_status[~required_status["non_empty"]]
missing_figures = figure_status[~figure_status["found"]]
bad_predictions = prediction_status[prediction_status["matches_test_rows"] == False] if not prediction_status.empty else pd.DataFrame()

print("\nSUMMARY")
print(f"Required files OK: {missing_required.empty}")
print(f"Figures OK: {missing_figures.empty}")
print(f"Prediction row counts OK: {bad_predictions.empty}")
print(f"Processed test rows: {test_rows}")

if not missing_required.empty:
    print("\nMissing/empty required files:")
    display(missing_required)

if not missing_figures.empty:
    print("\nMissing figures:")
    display(missing_figures)

if not bad_predictions.empty:
    print("\nPrediction files with wrong row counts:")
    display(bad_predictions)


Project root: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing
report.tex found: False -> C:\Users\ybenj\Downloads\report.tex

REQUIRED FILES


,path,exists,non_empty,size_bytes,rows_if_csv
0,outputs/data_summary.csv,True,True,955,15.0
1,outputs/label_distribution.csv,True,True,717,20.0
2,outputs/main_results.csv,True,True,4840,9.0
3,outputs/per_class_results.csv,True,True,44320,180.0
4,outputs/confusion_pairs.csv,True,True,401,15.0
5,outputs/misclassified_examples.csv,True,True,11890,10.0
6,outputs/hyperparameters.csv,True,True,1565,23.0
7,outputs/environment.json,True,True,688,NaN
8,outputs/report_artifact_manifest.json,True,True,2047,NaN
9,data/processed/dataset_summary.json,True,True,619,NaN



OPTIONAL / CONDITIONAL MODEL EVIDENCE FILES


,path,exists,non_empty,size_bytes,rows_if_csv
0,outputs/transformer_results.csv,True,True,672,1.0
1,outputs/transformer_predictions.csv,True,True,3263606,4732.0
2,outputs/qwen_results.csv,True,True,488,2.0
3,outputs/qwen_predictions.csv,True,True,299486,400.0
4,outputs/qwen_invalid_outputs.csv,True,True,167383,209.0
5,outputs/qwen_prompt_examples.txt,True,True,14554,NaN
6,results/transformer/runtime.json,True,True,458,NaN
7,results/transformer/training_args.json,True,True,520,NaN
8,results/transformer/training_log_history.json,True,True,2898,NaN
9,results/qwen/runtime.json,True,True,451,NaN



FIGURES


,figure_ref,found,found_at,size_bytes
0,figures/agentic_review_workflow.png,True,/content/drive/MyDrive/Colab Notebooks/Educati...,21264
1,figures/clause_length_distribution.png,True,/content/drive/MyDrive/Colab Notebooks/Educati...,38109
2,figures/confusion_matrix_best_model.png,True,/content/drive/MyDrive/Colab Notebooks/Educati...,114051
3,figures/label_distribution.png,True,/content/drive/MyDrive/Colab Notebooks/Educati...,69101
4,figures/model_comparison_macro_f1.png,True,/content/drive/MyDrive/Colab Notebooks/Educati...,78700
5,figures/qwen_invalid_predictions.png,True,/content/drive/MyDrive/Colab Notebooks/Educati...,31719



PREDICTION ROW COUNTS


,prediction_file,rows,matches_test_rows,expected_test_rows
0,outputs/predictions/distilbert_base_uncased_te...,4732,True,4732
1,outputs/predictions/linear_svm_test_prediction...,4732,True,4732
2,outputs/predictions/logistic_regression_test_p...,4732,True,4732
3,outputs/predictions/majority_baseline_test_pre...,4732,True,4732
4,outputs/predictions/multinomial_nb_test_predic...,4732,True,4732
5,outputs/predictions/random_train_distribution_...,4732,True,4732
6,outputs/predictions/random_uniform_test_predic...,4732,True,4732



SUMMARY
Required files OK: True
Figures OK: True
Prediction row counts OK: True
Processed test rows: 4732


In [ ]:

# Final report evidence audit cell
# Run before using the notebook outputs to fill report.tex.
 

# In Colab, set this manually if report.tex is uploaded or synced.
# REPORT_TEX_PATH = "/content/drive/MyDrive/.../report.tex"
REPORT_TEX_PATH = globals().get("REPORT_TEX_PATH", r"C:\Users\ybenj\Downloads\report.tex")

try:
    PROJECT_ROOT_CHECK = Path(paths.project_root)
except NameError:
    PROJECT_ROOT_CHECK = Path.cwd()

REPORT_TEX = Path(REPORT_TEX_PATH)

print(f"Project root: {PROJECT_ROOT_CHECK}")
print(f"report.tex found: {REPORT_TEX.exists()} -> {REPORT_TEX}")


def safe_display(title, df):
    print(f"\n{title}")
    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


def file_mtime(path):
    if not path.exists():
        return None
    return datetime.fromtimestamp(path.stat().st_mtime, tz=timezone.utc)


def count_jsonl(path):
    if not path.exists() or path.stat().st_size == 0:
        return None
    return sum(1 for line in path.open("r", encoding="utf-8") if line.strip())


def read_csv_safe(path):
    if not path.exists() or path.stat().st_size == 0:
        return None
    try:
        return pd.read_csv(path)
    except Exception as exc:
        print(f"Could not read CSV {path}: {type(exc).__name__}: {exc}")
        return None


def read_json_safe(path):
    if not path.exists() or path.stat().st_size == 0:
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Could not read JSON {path}: {type(exc).__name__}: {exc}")
        return None


def resolve_report_path(value):
    if pd.isna(value) or not str(value).strip():
        return None

    raw = str(value)
    direct = Path(raw)
    if direct.exists():
        return direct

    # Handle paths saved in Colab like /content/drive/.../results/...
    for marker in ["outputs/", "results/", "figures/", "data/processed/"]:
        if marker in raw.replace("\\", "/"):
            rel = raw.replace("\\", "/").split(marker, 1)[1]
            candidate = PROJECT_ROOT_CHECK / marker.rstrip("/") / rel
            if candidate.exists():
                return candidate

    candidate = PROJECT_ROOT_CHECK / raw
    if candidate.exists():
        return candidate

    return None


def safe_name(value):
    return re.sub(r"[^a-zA-Z0-9]+", "_", str(value).lower()).strip("_") or "item"


# 1. Required artifact checks

required_specs = {
    "outputs/data_summary.csv": ["metric", "value", "notes"],
    "outputs/label_distribution.csv": ["label", "label_id", "train_count", "validation_count", "test_count", "total_count"],
    "outputs/main_results.csv": ["model_family", "model_name", "sample_size", "accuracy", "macro_f1", "weighted_f1"],
    "outputs/per_class_results.csv": ["model_family", "model_name", "label", "precision", "recall", "f1_score", "support"],
    "outputs/confusion_pairs.csv": ["rank", "true_label", "predicted_label", "count"],
    "outputs/misclassified_examples.csv": ["text", "label", "predicted_label"],
    "outputs/hyperparameters.csv": ["component", "parameter", "value"],
    "outputs/environment.json": None,
    "outputs/leakage_audit.json": None,
    "outputs/report_artifact_manifest.json": None,
    "data/processed/dataset_summary.json": None,
    "data/processed/label_counts.json": None,
    "data/processed/label_names.txt": None,
    "data/processed/ledgar_train.jsonl": None,
    "data/processed/ledgar_validation.jsonl": None,
    "data/processed/ledgar_test.jsonl": None,
    "results/final_model_comparison.csv": ["model_family", "model_name", "sample_size", "accuracy", "macro_f1", "weighted_f1"],
}

artifact_rows = []
for rel_path, expected_cols in required_specs.items():
    path = PROJECT_ROOT_CHECK / rel_path
    exists = path.exists()
    non_empty = exists and path.is_file() and path.stat().st_size > 0
    columns_ok = True
    rows = None
    missing_cols = []

    if non_empty and rel_path.endswith(".csv"):
        df = read_csv_safe(path)
        if df is not None:
            rows = len(df)
            missing_cols = [c for c in expected_cols if c not in df.columns]
            columns_ok = not missing_cols
        else:
            columns_ok = False

    if non_empty and rel_path.endswith(".jsonl"):
        rows = count_jsonl(path)

    artifact_rows.append({
        "path": rel_path,
        "exists": exists,
        "non_empty": non_empty,
        "rows": rows,
        "columns_ok": columns_ok,
        "missing_columns": ", ".join(missing_cols),
        "modified_utc": file_mtime(path),
    })

artifact_status = pd.DataFrame(artifact_rows)
safe_display("1. REQUIRED ARTIFACTS", artifact_status)


# 2. Split consistency checks

split_paths = {
    "train": PROJECT_ROOT_CHECK / "data/processed/ledgar_train.jsonl",
    "validation": PROJECT_ROOT_CHECK / "data/processed/ledgar_validation.jsonl",
    "test": PROJECT_ROOT_CHECK / "data/processed/ledgar_test.jsonl",
}
split_counts = {split: count_jsonl(path) for split, path in split_paths.items()}

data_summary = read_csv_safe(PROJECT_ROOT_CHECK / "outputs/data_summary.csv")
summary_counts = {}

if data_summary is not None:
    summary_map = dict(zip(data_summary["metric"], data_summary["value"]))
    for split in ["train", "validation", "test"]:
        key = f"filtered_{split}_examples"
        try:
            summary_counts[split] = int(float(summary_map.get(key)))
        except Exception:
            summary_counts[split] = None

split_status = pd.DataFrame([
    {
        "split": split,
        "processed_jsonl_rows": split_counts.get(split),
        "data_summary_rows": summary_counts.get(split),
        "matches_data_summary": split_counts.get(split) == summary_counts.get(split),
    }
    for split in ["train", "validation", "test"]
])
safe_display("2. SPLIT CONSISTENCY", split_status)


# ----------------------------
# 3. Leakage audit check
# ----------------------------

leakage = read_json_safe(PROJECT_ROOT_CHECK / "outputs/leakage_audit.json")
leakage_rows = []

if leakage:
    after = leakage.get("cross_split_overlaps_after_deduplication", {})
    for pair, values in after.items():
        leakage_rows.append({
            "pair": pair,
            "text_overlap": values.get("text_overlap"),
            "text_label_overlap": values.get("text_label_overlap"),
            "ok_zero_overlap": values.get("text_overlap") == 0 and values.get("text_label_overlap") == 0,
        })

leakage_status = pd.DataFrame(leakage_rows)
safe_display("3. LEAKAGE AUDIT", leakage_status)


# ----------------------------
# 4. Metrics consistency checks
# ----------------------------

main_results = read_csv_safe(PROJECT_ROOT_CHECK / "outputs/main_results.csv")
final_comparison = read_csv_safe(PROJECT_ROOT_CHECK / "results/final_model_comparison.csv")

metric_consistency_rows = []

if main_results is not None and final_comparison is not None:
    keys = ["model_family", "model_name"]
    metric_cols = ["sample_size", "accuracy", "macro_f1", "weighted_f1"]
    merged = main_results[keys + metric_cols].merge(
        final_comparison[keys + metric_cols],
        on=keys,
        suffixes=("_outputs", "_results"),
        how="outer",
        indicator=True,
    )

    for _, row in merged.iterrows():
        ok = row["_merge"] == "both"
        diffs = []
        if ok:
            for col in metric_cols:
                left = row[f"{col}_outputs"]
                right = row[f"{col}_results"]
                if pd.isna(left) and pd.isna(right):
                    continue
                if col == "sample_size":
                    same = int(left) == int(right)
                else:
                    same = abs(float(left) - float(right)) < 1e-9
                if not same:
                    ok = False
                    diffs.append(col)

        metric_consistency_rows.append({
            "model_family": row.get("model_family"),
            "model_name": row.get("model_name"),
            "present_in_both": row["_merge"] == "both",
            "metrics_match": ok,
            "different_columns": ", ".join(diffs),
        })

metric_consistency = pd.DataFrame(metric_consistency_rows)
safe_display("4. METRIC CONSISTENCY: outputs/main_results.csv vs results/final_model_comparison.csv", metric_consistency)


# ----------------------------
# 5. Classification report/confusion matrix existence
# ----------------------------

report_rows = []

if main_results is not None:
    for _, row in main_results.iterrows():
        sample_size = row.get("sample_size")
        macro_f1 = row.get("macro_f1")
        is_completed = pd.notna(macro_f1) and pd.notna(sample_size) and int(sample_size) > 0

        report_path = resolve_report_path(row.get("classification_report_path"))
        cm_path = resolve_report_path(row.get("confusion_matrix_path"))

        report_rows.append({
            "model_name": row.get("model_name"),
            "completed_result": is_completed,
            "classification_report_found": bool(report_path) if is_completed else "not_required_if_skipped",
            "confusion_matrix_found": bool(cm_path) if is_completed else "not_required_if_skipped",
            "classification_report_path": str(report_path) if report_path else "",
            "confusion_matrix_path": str(cm_path) if cm_path else "",
        })

report_status = pd.DataFrame(report_rows)
safe_display("5. REPORT + CONFUSION MATRIX EVIDENCE", report_status)


# ----------------------------
# 6. Prediction file row counts
# ----------------------------

test_rows = split_counts.get("test")
prediction_rows = []
prediction_dir = PROJECT_ROOT_CHECK / "outputs/predictions"

if prediction_dir.exists():
    for pred_path in sorted(prediction_dir.glob("*_test_predictions.jsonl")):
        rows = count_jsonl(pred_path)
        prediction_rows.append({
            "prediction_file": str(pred_path.relative_to(PROJECT_ROOT_CHECK)),
            "rows": rows,
            "expected_test_rows": test_rows,
            "matches_test_rows": rows == test_rows,
            "modified_utc": file_mtime(pred_path),
        })

prediction_status = pd.DataFrame(prediction_rows)
safe_display("6. PREDICTION ROW COUNTS", prediction_status)


# ----------------------------
# 7. Skipped transformer/Qwen guard
# ----------------------------

guard_rows = []

if main_results is not None:
    for model_group, mask in {
        "transformer": main_results["model_family"].astype(str).str.contains("transformer", case=False, na=False)
                       | main_results["model_name"].astype(str).str.contains("bert|distilbert|legal", case=False, na=False),
        "qwen": main_results["model_name"].astype(str).str.contains("qwen", case=False, na=False)
                | main_results["model_family"].astype(str).str.contains("qwen|llm", case=False, na=False),
    }.items():
        subset = main_results[mask]
        if subset.empty:
            guard_rows.append({
                "model_group": model_group,
                "present": False,
                "real_result_rows": 0,
                "skipped_or_empty_rows": 0,
                "safe_to_claim_results": False,
            })
        else:
            real = subset[pd.to_numeric(subset["sample_size"], errors="coerce").fillna(0).gt(0) & subset["macro_f1"].notna()]
            guard_rows.append({
                "model_group": model_group,
                "present": True,
                "real_result_rows": len(real),
                "skipped_or_empty_rows": len(subset) - len(real),
                "safe_to_claim_results": len(real) > 0,
            })

skipped_guard = pd.DataFrame(guard_rows)
safe_display("7. SKIPPED MODEL GUARD", skipped_guard)


# ----------------------------
# 8. Figure reference checks
# ----------------------------

standard_figures = [
    "figures/label_distribution.png",
    "figures/clause_length_distribution.png",
    "figures/agentic_review_workflow.png",
    "figures/model_comparison_macro_f1.png",
    "figures/confusion_matrix_best_model.png",
    "figures/qwen_invalid_predictions.png",
]

figure_refs_from_tex = []
todo_rows = []

if REPORT_TEX.exists():
    tex = REPORT_TEX.read_text(encoding="utf-8", errors="ignore")
    figure_refs_from_tex = re.findall(r"\\includegraphics(?:\[[^\]]*\])?\{([^}]+)\}", tex)

    for line_no, line in enumerate(tex.splitlines(), 1):
        for todo in re.findall(r"\\todo\{([^}]*)\}", line):
            todo_rows.append({
                "line": line_no,
                "todo": todo,
                "likely_evidence": (
                    "outputs/main_results.csv" if any(x in todo.lower() for x in ["result", "score", "model", "f1"]) else
                    "outputs/data_summary.csv" if any(x in todo.lower() for x in ["example", "length", "train", "validation", "test"]) else
                    "outputs/per_class_results.csv" if "class" in todo.lower() or "categor" in todo.lower() else
                    "outputs/confusion_pairs.csv" if "label" in todo.lower() else
                    "manual review needed"
                ),
            })

figure_refs = sorted(set(standard_figures + figure_refs_from_tex))

qwen_real = False
if main_results is not None:
    qwen_mask = main_results["model_name"].astype(str).str.contains("qwen", case=False, na=False)
    qwen_subset = main_results[qwen_mask]
    if not qwen_subset.empty:
        qwen_real = bool(
            pd.to_numeric(qwen_subset["sample_size"], errors="coerce").fillna(0).gt(0).any()
            and qwen_subset["macro_f1"].notna().any()
        )

figure_rows = []
for ref in figure_refs:
    ref_path = Path(ref)
    candidates = [
        PROJECT_ROOT_CHECK / ref,
        PROJECT_ROOT_CHECK / "outputs" / ref,
    ]
    if len(ref_path.parts) >= 2 and ref_path.parts[0] == "figures":
        candidates.append(PROJECT_ROOT_CHECK / "outputs" / "figures" / ref_path.name)

    existing = [p for p in candidates if p.exists() and p.is_file() and p.stat().st_size > 0]
    is_qwen_invalid = ref_path.name == "qwen_invalid_predictions.png"

    figure_rows.append({
        "figure_ref": ref,
        "found": bool(existing),
        "conditional_qwen_figure": is_qwen_invalid,
        "qwen_real_result_available": qwen_real,
        "ok_for_pipeline": bool(existing) or (is_qwen_invalid and not qwen_real),
        "report_edit_warning": "remove/keep TODO if Qwen skipped" if is_qwen_invalid and not qwen_real else "",
        "found_at": str(existing[0]) if existing else "",
        "size_bytes": existing[0].stat().st_size if existing else 0,
    })

figure_status = pd.DataFrame(figure_rows)
safe_display("8. FIGURE REFERENCES", figure_status)


# ----------------------------
# 9. TODO evidence map
# ----------------------------

todo_status = pd.DataFrame(todo_rows)
safe_display("9. REPORT TODO EVIDENCE MAP", todo_status)


# ----------------------------
# 10. Final strict summary
# ----------------------------

critical_failures = []

if not artifact_status["non_empty"].all():
    critical_failures.append("Missing or empty required artifacts.")

if not artifact_status["columns_ok"].all():
    critical_failures.append("Some required CSVs are missing expected columns.")

if not split_status["matches_data_summary"].all():
    critical_failures.append("Processed split counts do not match outputs/data_summary.csv.")

if not leakage_status.empty and not leakage_status["ok_zero_overlap"].all():
    critical_failures.append("Leakage audit still shows cross-split overlaps.")

if not metric_consistency.empty and not metric_consistency["metrics_match"].all():
    critical_failures.append("outputs/main_results.csv and results/final_model_comparison.csv disagree.")

if not prediction_status.empty and not prediction_status["matches_test_rows"].all():
    critical_failures.append("One or more test prediction files do not match processed test row count.")

completed_missing_reports = report_status[
    (report_status["completed_result"] == True)
    & (
        (report_status["classification_report_found"] != True)
        | (report_status["confusion_matrix_found"] != True)
    )
]
if not completed_missing_reports.empty:
    critical_failures.append("A completed model is missing a classification report or confusion matrix.")

blocking_figures = figure_status[figure_status["ok_for_pipeline"] != True]
if not blocking_figures.empty:
    critical_failures.append("One or more required non-conditional figures are missing.")

print("\nFINAL SUMMARY")
print(f"Critical failures: {len(critical_failures)}")
for failure in critical_failures:
    print(f"- {failure}")

if not critical_failures:
    print("PASS: report-facing evidence looks internally consistent.")
else:
    print("FAIL: fix the issues above before asking Codex to fill report.tex.")


Project root: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing
report.tex found: False -> C:\Users\ybenj\Downloads\report.tex

1. REQUIRED ARTIFACTS


,path,exists,non_empty,rows,columns_ok,missing_columns,modified_utc
0,outputs/data_summary.csv,True,True,15.0,True,,2026-05-02 23:25:20+00:00
1,outputs/label_distribution.csv,True,True,20.0,True,,2026-05-02 23:25:21+00:00
2,outputs/main_results.csv,True,True,9.0,True,,2026-05-02 23:25:23+00:00
3,outputs/per_class_results.csv,True,True,180.0,True,,2026-05-02 23:25:38+00:00
4,outputs/confusion_pairs.csv,True,True,15.0,True,,2026-05-02 23:25:38+00:00
5,outputs/misclassified_examples.csv,True,True,10.0,True,,2026-05-02 23:25:39+00:00
6,outputs/hyperparameters.csv,True,True,23.0,True,,2026-05-02 23:25:37+00:00
7,outputs/environment.json,True,True,NaN,True,,2026-05-02 23:25:50+00:00
8,outputs/leakage_audit.json,True,True,NaN,True,,2026-05-02 23:12:09+00:00
9,outputs/report_artifact_manifest.json,True,True,NaN,True,,2026-05-02 23:25:50+00:00



2. SPLIT CONSISTENCY


,split,processed_jsonl_rows,data_summary_rows,matches_data_summary
0,train,28587,28587,True
1,validation,4670,4670,True
2,test,4732,4732,True



3. LEAKAGE AUDIT


...


8. FIGURE REFERENCES


,figure_ref,found,conditional_qwen_figure,qwen_real_result_available,ok_for_pipeline,report_edit_warning,found_at,size_bytes
0,figures/agentic_review_workflow.png,True,False,True,True,,/content/drive/MyDrive/Colab Notebooks/Educati...,21264
1,figures/clause_length_distribution.png,True,False,True,True,,/content/drive/MyDrive/Colab Notebooks/Educati...,38109
2,figures/confusion_matrix_best_model.png,True,False,True,True,,/content/drive/MyDrive/Colab Notebooks/Educati...,114051
3,figures/label_distribution.png,True,False,True,True,,/content/drive/MyDrive/Colab Notebooks/Educati...,69101
4,figures/model_comparison_macro_f1.png,True,False,True,True,,/content/drive/MyDrive/Colab Notebooks/Educati...,78700
5,figures/qwen_invalid_predictions.png,True,True,True,True,,/content/drive/MyDrive/Colab Notebooks/Educati...,31719



9. REPORT TODO EVIDENCE MAP


""



FINAL SUMMARY
Critical failures: 0
PASS: report-facing evidence looks internally consistent.
